# ⚡ Samtidige Agentworkflows med GitHub-modeller (Python)

## 📋 Avanceret Parallel Behandlingsvejledning

Denne notebook demonstrerer **samtidige workflow-mønstre** ved brug af Microsoft Agent Framework. Du vil lære at bygge højtydende, parallelle processeringsworkflows, hvor flere AI-agenter kører samtidig, hvilket dramatisk forbedrer gennemløb og muliggør sofistikerede multithreadede forretningsprocesser.

## 🎯 Læringsmål

### 🚀 **Grundlæggende om Samtidig Behandling**
- **Parallelt Agentkørsel**: Kør flere agenter samtidig for maksimal effektivitet
- **Workfloworkestrering**: Koordinér samtidige operationer samtidig med data konsistens
- **Ydelsesoptimering**: Opnå betydelig hastighedsforøgelse gennem parallel behandling
- **Ressourcestyring**: Udnyt AI-modelressourcer effektivt på tværs af samtidige operationer

### 🏗️ **Avancerede Samtidighedsmønstre**
- **Fork-Join Behandling**: Del arbejdet mellem flere agenter og saml resultater
- **Pipelineparallelisme**: Overlappende udførelsesfaser for kontinuerligt gennemløb
- **Load Balancing**: Fordel arbejde jævnt mellem tilgængelige agentressourcer
- **Synkroniseringspunkter**: Koordinér samtidige agenter på kritiske workflowtrin

### 🏢 **Virksomhedsorienterede Concurrent Applikationer**
- **Dokumentbehandling med stort volumen**: Behandl flere dokumenter samtidigt
- **Realtime Indholdsanalyse**: Samtidig analyse af indkommende datastrømme
- **Batch-Processeringsoptimering**: Maksimer gennemløb for stor-skala operationer
- **Multimodal Analyse**: Parallel behandling af forskellige indholdstyper (tekst, billeder, data)

## ⚙️ Forudsætninger & Opsætning

### 📦 **Nødvendige Afhængigheder**

Installer Agent Framework med samtidige workflow-funktioner:

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub Models Konfiguration**

**Miljøopsætning (.env-fil):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**Overvejelser ved Samtidig Behandling:**
- **Rategrænser**: Overvåg GitHub Models API ratebegrænsninger for samtidige anmodninger
- **Ressourceforbrug**: Overvej hukommelse og CPU-forbrug med flere samtidige agenter
- **Fejlhåndtering**: Implementér robust fejlgendannelse for parallelle operationer

### 🏗️ **Samtidig Workflow-arkitektur**

```mermaid
graph TD
    A[Arbejdsproces start] --> B[Parallelt udførsel]
    B --> C[Agent Pulje 1]
    B --> D[Agent Pulje 2]
    B --> E[Agent Pulje 3]
    C --> F[Resultat samling]
    D --> F
    E --> F
    F --> G[Endelig output]
    
    H[GitHub Models API] --> C
    H --> D
    H --> E
```

**Nøglefordele:**
- **⚡ Ydelse**: Betydelig hastighedsforøgelse gennem parallel udførelse
- **📈 Skalérbarhed**: Håndter øget arbejdsmængde uden proportional tidsforøgelse
- **🔄 Effektivitet**: Bedre udnyttelse af tilgængelige beregningsressourcer
- **🎯 Gennemløb**: Behandle mere arbejde på samme tid

## 🎨 **Designmønstre for Samtidige Workflows**

### 🔍 **Forsknings- & Analysepipeline**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Databehandlingsworkflow**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Indholdsskabelsespipeline**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Flertrinsbehandling**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Virksomhedens Performancefordele**

### ⚡ **Gennemløbsoptimering**
- **Parallelt Kørsel**: Flere agenter arbejder samtidig
- **Ressourceudnyttelse**: Maksimal effektivitet af tilgængelig AI-modellkapacitet
- **Tidsreduktion**: Betydelig nedsættelse af samlet behandlingstid
- **Skalerbar Arkitektur**: Tilføj nemt flere samtidige agenter efter behov

### 🛡️ **Pålidelighed & Robusthed**
- **Fejltolerance**: Individuelle agentfejl stopper ikke hele workflowet
- **Fejisolation**: Problemer i én samtidig gren påvirker ikke andre
- **Graciøs Degeneration**: Systemet fortsætter drift selv med reduceret agentkapacitet
- **Genopretningsmekanismer**: Automatisk genforsøg og fejlhåndtering af fejlede operationer

### 📊 **Overvågning & Observabilitet**
- **Samtidig Udførelsessporing**: Overvåg fremskridt i alle parallelle operationer
- **Ydelsesmålinger**: Mål hastighedsforøgelse og effektivitetsgevinster
- **Ressourcebrugsanalyse**: Optimer samtidige agentallokeringer
- **Flaskehalsidentifikation**: Find og løs performancebegrænsninger

Lad os bygge højtydende samtidige AI-workflows! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
